# 01 — Bronze Ingestion

**Purpose:** Raw source ingestion to Bronze Delta tables.  
**Runtime:** Microsoft Fabric PySpark with attached default Lakehouse.  
**Status:** Provided as source; requires execution and verification in your Fabric tenant.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

# Attach a default Lakehouse to this Fabric notebook before running.
# Relative Files paths are resolved against the default Lakehouse in Spark notebooks.
BASE = 'Files/source/ecommerce'
SCHEMAS = {
    'customers': 'customer_id STRING, customer_name STRING, region STRING, signup_date STRING',
    'products': 'product_id STRING, product_name STRING, category STRING, list_price STRING',
    'orders': 'order_id STRING, customer_id STRING, order_date STRING, order_status STRING',
    'order_items': 'order_item_id STRING, order_id STRING, product_id STRING, quantity STRING, unit_price STRING, discount_pct STRING',
}
for name, schema in SCHEMAS.items():
    df=(spark.read.option('header','true').schema(schema).csv(f'{BASE}/{name}.csv')
        .withColumn('_ingested_at_utc',F.current_timestamp())
        .withColumn('_source_file',F.input_file_name()))
    (df.write.format('delta').mode('overwrite').option('overwriteSchema','true')
       .saveAsTable(f'bronze_{name}'))
    print(f'bronze_{name}: {df.count():,} rows')
